In [1]:
!python3 --version

Python 3.8.10


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('reviews.csv')

In [10]:
df.shape

(10548, 14)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10548 entries, 0 to 10547
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   submission_id         10527 non-null  object 
 1   paper_title           10548 non-null  object 
 2   track                 10548 non-null  object 
 3   author_ids            10548 non-null  object 
 4   reviewer_id           10524 non-null  object 
 5   assignment_timestamp  10548 non-null  object 
 6   review_timestamp      10548 non-null  object 
 7   soundness             10548 non-null  object 
 8   excitement            10548 non-null  int64  
 9   confidence            10521 non-null  float64
 10  overall_score         10481 non-null  float64
 11  metareview_score      10521 non-null  float64
 12  review_text           10521 non-null  object 
 13  recommendation        10521 non-null  object 
dtypes: float64(3), int64(1), object(10)
memory usage: 1.1+ MB


In [26]:
df.dtypes

submission_id            object
paper_title              object
track                    object
author_ids               object
reviewer_id              object
assignment_timestamp     object
review_timestamp         object
soundness                object
excitement                int64
confidence              float64
overall_score           float64
metareview_score        float64
review_text              object
recommendation           object
dtype: object

In [25]:
df.isnull().sum()

submission_id           21
paper_title              0
track                    0
author_ids               0
reviewer_id             24
assignment_timestamp     0
review_timestamp         0
soundness                0
excitement               0
confidence              27
overall_score           67
metareview_score        27
review_text             27
recommendation          27
dtype: int64

In [27]:
df["soundness"].value_counts(dropna=False)

soundness
3       4496
2       3315
4       1766
1        713
5        226
high      32
Name: count, dtype: int64

#### SCHEMA LEVEL VALIDATION

##### Validate submission_id and reviewer_id

In [30]:
valid_submission_id = (
    df["submission_id"].notna()
    & df["submission_id"].astype(str).str.strip().ne("")
)

valid_reviewer_id = (
    df["reviewer_id"].notna()
    & df["reviewer_id"].astype(str).str.strip().ne("")
)

##### Validate track and recommendation

In [46]:
allowed_tracks = {
    "MainConference",
    "Findings",
    "Workshop",
    "Demo"
}

allowed_recommendations = {
    "Strong Accept",
    "Accept",
    "Borderline",
    "Reject",
    "Strong Reject"
}

valid_track = df['track'].isin(allowed_tracks)
valid_recommendation = df["recommendation"].isin(
    allowed_recommendations
)

In [47]:
(~valid_track).sum()

0

In [48]:
df.loc[~valid_track, ["submission_id", "track"]]

,submission_id,track


In [49]:
(~valid_recommendation).sum()

27

In [62]:
#Inspect malformity

df.loc[~valid_recommendation, ["submission_id", "recommendation"]].head(2)

,submission_id,recommendation
174,S00757,NaN
575,S02505,NaN


In [51]:
assignment_dt = pd.to_datetime(
    df["assignment_timestamp"],
    errors="coerce",
    utc=True
)

review_dt = pd.to_datetime(
    df["review_timestamp"],
    errors="coerce",
    utc=True
)

In [52]:
valid_assignment_timestamp = assignment_dt.notna()
valid_review_timestamp = review_dt.notna()

In [53]:
(~valid_assignment_timestamp).sum()

37

In [54]:
(~valid_review_timestamp).sum()

31

In [57]:
df.loc[~valid_assignment_timestamp, ["submission_id", "assignment_timestamp"]].head(2)

,submission_id,assignment_timestamp
328,S02374,2025/03/14
996,S00204,2025/03/14


In [61]:
df.loc[~valid_review_timestamp, ["submission_id", "review_timestamp"]].head(2)

,submission_id,review_timestamp
569,S01360,yesterday
612,S00895,yesterday


##### Validate soundness, excitement and confidence

In [63]:
def valid_integer_score(label, lower=1, upper=5):
    parsed = pd.to_numeric(label, errors="coerce")
    
    return (
        parsed.notna()
        & (parsed % 1 == 0)
        & parsed.between(lower, upper)
    )

In [64]:
valid_soundness = valid_integer_score(df["soundness"])
valid_excitement = valid_integer_score(df["excitement"])
valid_confidence = valid_integer_score(df["confidence"])

In [72]:
(~valid_soundness).sum()

32

In [73]:
(~valid_excitement).sum()

0

In [74]:
(~valid_confidence).sum()

65

In [71]:
df['soundness'].value_counts()

soundness
3       4496
2       3315
4       1766
1        713
5        226
high      32
Name: count, dtype: int64

In [68]:
df.loc[~valid_soundness, ["submission_id", "soundness"]]

,submission_id,soundness
260,S01847,high
535,S02590,high
664,S02939,high
770,S00921,high
1749,S00877,high
1962,S00523,high
2305,S02168,high
3191,S01685,high
3356,S03408,high
3491,S01392,high


In [75]:
df.loc[~valid_excitement, ["submission_id", "excitement"]]

,submission_id,excitement


In [76]:
df.loc[~valid_confidence, ["submission_id", "confidence"]]

,submission_id,confidence
174,S00757,NaN
441,S01087,99.0
575,S02505,NaN
580,S01355,NaN
631,S00405,NaN
...,...,...
9663,S00451,7.0
9684,S00781,99.0
9871,S02488,NaN
10023,S03184,0.0
